In [32]:
import pandas as pd
import numpy as np
job_exposure = pd.read_csv("data/job_exposure.csv")
task_penetration = pd.read_csv("data/task_penetration.csv")

In [33]:
print(job_exposure.shape)
print(task_penetration.shape)

(756, 3)
(17998, 2)


In [34]:
job_exposure.head()

,occ_code,title,observed_exposure
0,11-1011,Chief Executives,0.0333
1,11-1021,General and Operations Managers,0.1378
2,11-1031,Legislators,0.0000
3,11-2011,Advertising and Promotions Managers,0.1731
4,11-2021,Marketing Managers,0.3195


In [35]:
task_penetration.head()

,task,penetration
0,Accept and check containers of mail from large...,0.0
1,Accept and check containers of mail or parcels...,0.0
2,Accept credit applications and verify credit r...,0.0
3,Accept music requests from event guests.,0.0
4,"Accept payment from customers, and make change...",0.0


## 1. State the Project Vision and Question

**Status: Done**

**Project Vision:**  
Our project aims to help users understand how different occupations and job tasks are exposed to AI. The dashboard will allow users to compare jobs and explore which occupations or tasks may be more affected by AI.

**Business Question:**  
Which occupations and job tasks show higher levels of AI exposure?

**Main Metrics:**  
- `observed_exposure` for occupations
- `penetration` for job tasks

Higher values of these metrics indicate greater observed AI exposure or penetration.

## 2. Define One Row

**Status: Done**

The two datasets have different units of observation.

**`job_exposure`:**
- One row represents one occupation.
- The dataset contains 756 rows.
- `occ_code` is the unique identifier.
- All 756 `occ_code` values are nonmissing and unique.

**`task_penetration`:**
- One row represents one job task record and its penetration value.
- The dataset contains 17,998 rows.
- `task` was initially considered as a possible identifier, but it is not unique.
- There are 17,992 distinct original task descriptions.
- 8 rows are involved in repeated task descriptions.

Therefore, `occ_code` uniquely identifies rows in `job_exposure`, while `task` does not uniquely identify every row in the original `task_penetration` dataset.

In [36]:
print("job_exposure")
print("Rows:", len(job_exposure))
print("Unique occ_code:", job_exposure["occ_code"].nunique())
print("Missing occ_code:", job_exposure["occ_code"].isna().sum())
print("occ_code is unique:", job_exposure["occ_code"].is_unique)

print("\ntask_penetration")
print("Rows:", len(task_penetration))
print("Unique tasks:", task_penetration["task"].nunique())
print("Missing tasks:", task_penetration["task"].isna().sum())
print("task is unique:", task_penetration["task"].is_unique)

job_exposure
Rows: 756
Unique occ_code: 756
Missing occ_code: 0
occ_code is unique: True

task_penetration
Rows: 17998
Unique tasks: 17992
Missing tasks: 0
task is unique: False


## 3. Preserve the Source

**Status: Done**

The original datasets were obtained from Anthropic's Economic Index dataset on Hugging Face, specifically from the `labor_market_impacts` folder.

**Data Source:** Anthropic Economic Index  
**Source Platform:** Hugging Face  
**Folder:** `labor_market_impacts`  
**Files used:**
- `job_exposure.csv`
- `task_penetration.csv`

**Download date:** October 2026

The original CSV files are stored in the `data` folder:

- `data/job_exposure.csv`
- `data/task_penetration.csv`

The original files are kept unchanged. All inspection and cleaning steps are performed in `cleaning_ella.ipynb` after loading the original files with pandas.

The original datasets will not be overwritten. Any cleaned data will be saved separately so that the cleaning process can be reproduced from the original source files.

In [37]:
pd.read_csv("data/job_exposure.csv")
pd.read_csv("data/task_penetration.csv")

,task,penetration
0,Accept and check containers of mail from large...,0.0000
1,Accept and check containers of mail or parcels...,0.0000
2,Accept credit applications and verify credit r...,0.0000
3,Accept music requests from event guests.,0.0000
4,"Accept payment from customers, and make change...",0.0000
...,...,...
17993,"Write, prepare, and deliver statements for the...",0.0000
17994,"Write, review, or comment on documents, such a...",0.0000
17995,"Write, review, or execute plans for testing ne...",0.0000
17996,"Write, review, or maintain engineering documen...",0.8133


### 4. Inspect Columns and Types

**Status: Done**

All five columns have appropriate data types, so no type conversions are needed.

- `occ_code` is stored as text because it is an identifier and its formatting should be preserved.
- `title` and `task` are text fields.
- `observed_exposure` and `penetration` are numeric (`float64`).
- Both numeric columns had 0 conversion failures and 0 infinite values.

No data type changes were needed.

In [38]:
job_numeric = pd.to_numeric(
    job_exposure["observed_exposure"],
    errors="coerce"
)

task_numeric = pd.to_numeric(
    task_penetration["penetration"],
    errors="coerce"
)

print("observed_exposure conversion failures:",
      job_numeric.isna().sum())

print("penetration conversion failures:",
      task_numeric.isna().sum())

print("observed_exposure infinite values:",
      np.isinf(job_numeric).sum())

print("penetration infinite values:",
      np.isinf(task_numeric).sum())

observed_exposure conversion failures: 0
penetration conversion failures: 0
observed_exposure infinite values: 0
penetration infinite values: 0


## 5. Standardize Text

**Status: Done**

I checked the text columns for formatting inconsistencies, including leading/trailing whitespace and capitalization differences.

In `task_penetration`, one group of task descriptions differs only in the capitalization of `"Web"` versus `"web"`.

A full-column check found:
- 0 rows with leading or trailing whitespace.
- 1 repeated-text group after trimming whitespace and ignoring capitalization.
- 8 rows in this group.
- All 8 rows have the same `penetration` value of 0.7263.
- No additional task descriptions became duplicates after trimming whitespace and ignoring capitalization.

**Cleaning Rule:**  
Treat these 8 descriptions as the same task because the only difference is capitalization and their penetration values are identical. Standardize this capitalization difference before handling the duplicate records.

Other text values will remain unchanged because no additional formatting inconsistencies requiring cleaning were identified.

In [39]:
print(
    "Task rows with leading/trailing whitespace:",
    (task_penetration["task"] != task_penetration["task"].str.strip()).sum()
)

task_key = task_penetration["task"].str.strip().str.casefold()

case_duplicate_mask = task_key.duplicated(keep=False)

print(
    "Rows involved in case-insensitive repeated tasks:",
    case_duplicate_mask.sum()
)

task_penetration.loc[case_duplicate_mask]

Task rows with leading/trailing whitespace: 0
Rows involved in case-insensitive repeated tasks: 8


,task,penetration
17889,Write interesting and effective press releases...,0.7263
17890,Write interesting and effective press releases...,0.7263
17891,Write interesting and effective press releases...,0.7263
17892,Write interesting and effective press releases...,0.7263
17893,Write interesting and effective press releases...,0.7263
17894,Write interesting and effective press releases...,0.7263
17895,Write interesting and effective press releases...,0.7263
17896,Write interesting and effective press releases...,0.7263


## 6. Investigate Duplicates

**Status: Done**

Exact duplicate rows and repeated identifiers/descriptions were investigated separately.

**`job_exposure`:**
- 756 total rows.
- 756 distinct `occ_code` values.
- 0 repeated `occ_code` values.
- No duplicate occupation identifiers were found.

**`task_penetration`:**
- 17,998 total rows.
- 17,992 distinct original task descriptions.
- 8 rows are involved in repeated task descriptions.
- The 8 rows contain two versions of the same task description, differing only in `"Web"` versus `"web"`.
- All 8 records have the same `penetration` value of 0.7263.
- Within the two original text versions, there are 6 exact duplicate rows beyond the first occurrence.
- After ignoring the capitalization difference, all 8 records represent the same task.

**Cleaning Rule:**  
Standardize the capitalization difference and keep one copy of the repeated task. The remaining redundant copies will be removed.

After standardization, the 8 records should become 1 record, resulting in 7 redundant rows being removed.

**Check:**  
After cleaning, verify that only one record for this task remains and that its `penetration` value is still 0.7263.

In [40]:
print("job_exposure exact duplicate rows:",
      job_exposure.duplicated().sum())

print("job_exposure repeated occ_code:",
      job_exposure["occ_code"].duplicated().sum())

print("task_penetration exact duplicate rows:",
      task_penetration.duplicated().sum())

print("task_penetration repeated task descriptions:",
      task_penetration["task"].duplicated().sum())

repeated_tasks = task_penetration.loc[
    task_penetration["task"].duplicated(keep=False)
].sort_values("task", kind="stable")

repeated_tasks

job_exposure exact duplicate rows: 0
job_exposure repeated occ_code: 0
task_penetration exact duplicate rows: 6
task_penetration repeated task descriptions: 6


,task,penetration
17889,Write interesting and effective press releases...,0.7263
17890,Write interesting and effective press releases...,0.7263
17891,Write interesting and effective press releases...,0.7263
17892,Write interesting and effective press releases...,0.7263
17893,Write interesting and effective press releases...,0.7263
17894,Write interesting and effective press releases...,0.7263
17895,Write interesting and effective press releases...,0.7263
17896,Write interesting and effective press releases...,0.7263


### 7. Handle Missing Data

**Status: Done**

I checked all columns in both datasets for missing values.

- `job_exposure`: 0 missing values in `occ_code`, `title`, and `observed_exposure`.
- `task_penetration`: 0 missing values in `task` and `penetration`.
- The missing percentage was 0.00% for every column.

Since no missing values were found, no values needed to be filled, dropped, or otherwise modified.

In [41]:
for name, df in [
    ("job_exposure", job_exposure),
    ("task_penetration", task_penetration),
]:
    print(f"\n{name}")
    
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": df.isna().mean() * 100
    })
    
    display(missing)


job_exposure


,missing_count,missing_percentage
occ_code,0,0.0
title,0,0.0
observed_exposure,0,0.0



task_penetration


,missing_count,missing_percentage
task,0,0.0
penetration,0,0.0


## 8. Check Suspicious Values

**Status: Done**

I checked both datasets for invalid and unusual values.

**`job_exposure`:**
- `observed_exposure` ranges from 0.0 to 0.7451.
- 0 values are below 0.
- 0 values are above 1.
- All `occ_code` values follow the expected `NN-NNNN` format.
- There are 0 blank occupation titles.

**`task_penetration`:**
- `penetration` ranges from 0.0 to 1.0.
- 0 values are below 0.
- 0 values are above 1.
- There are 0 blank task descriptions.

I also inspected records with the lowest and highest values. For example, some occupations have an `observed_exposure` of 0.0, while Computer Programmers have the highest observed exposure of 0.7451. Some tasks also have penetration values of 0.0 or 1.0.

These extreme values are within the valid 0–1 range and correspond to valid records, so they were treated as legitimate observations rather than errors.

**Cleaning Decision:** No records were removed or changed based on unusual numeric values.

In [42]:
print("job_exposure:")
print("Minimum observed_exposure:", job_exposure["observed_exposure"].min())
print("Maximum observed_exposure:", job_exposure["observed_exposure"].max())
print("Values below 0:", (job_exposure["observed_exposure"] < 0).sum())
print("Values above 1:", (job_exposure["observed_exposure"] > 1).sum())

print("\ntask_penetration:")
print("Minimum penetration:", task_penetration["penetration"].min())
print("Maximum penetration:", task_penetration["penetration"].max())
print("Values below 0:", (task_penetration["penetration"] < 0).sum())
print("Values above 1:", (task_penetration["penetration"] > 1).sum())

job_exposure:
Minimum observed_exposure: 0.0
Maximum observed_exposure: 0.7451
Values below 0: 0
Values above 1: 0

task_penetration:
Minimum penetration: 0.0
Maximum penetration: 1.0
Values below 0: 0
Values above 1: 0


In [43]:
invalid_occ_code = ~job_exposure["occ_code"].str.match(r"^\d{2}-\d{4}$")

blank_titles = job_exposure["title"].str.strip().eq("")
blank_tasks = task_penetration["task"].str.strip().eq("")

print("Invalid occ_code formats:", invalid_occ_code.sum())
print("Blank titles:", blank_titles.sum())
print("Blank tasks:", blank_tasks.sum())

Invalid occ_code formats: 0
Blank titles: 0
Blank tasks: 0


In [44]:
print("Lowest observed exposure:")
display(job_exposure.nsmallest(5, "observed_exposure"))

print("Highest observed exposure:")
display(job_exposure.nlargest(5, "observed_exposure"))

print("Lowest task penetration:")
display(task_penetration.nsmallest(5, "penetration"))

print("Highest task penetration:")
display(task_penetration.nlargest(5, "penetration"))

Lowest observed exposure:


,occ_code,title,observed_exposure
2,11-1031,Legislators,0.0
7,11-3012,Administrative Services Managers,0.0
14,11-3111,Compensation and Benefits Managers,0.0
15,11-3121,Human Resources Managers,0.0
17,11-9013,"Farmers, Ranchers, and Other Agricultural Mana...",0.0


Highest observed exposure:


,occ_code,title,observed_exposure
73,15-1251,Computer Programmers,0.7451
461,43-4051,Customer Service Representatives,0.7011
488,43-9021,Data Entry Keyers,0.6707
330,29-2072,Medical Records Specialists,0.6674
49,13-1161,Market Research Analysts and Marketing Special...,0.6483


Lowest task penetration:


,task,penetration
0,Accept and check containers of mail from large...,0.0
1,Accept and check containers of mail or parcels...,0.0
2,Accept credit applications and verify credit r...,0.0
3,Accept music requests from event guests.,0.0
4,"Accept payment from customers, and make change...",0.0


Highest task penetration:


,task,penetration
341,Advise customers on use and care of merchandise.,1.0
404,Advise students on academic and vocational cur...,1.0
501,Analyze characteristics of animals to identify...,1.0
522,Analyze data to determine answers to questions...,1.0
531,Analyze developments in specific field to dete...,1.0


## 9. Verify Merges

**Status: Not applicable**

The two datasets were not merged because they do not contain a valid shared key.

- `job_exposure` contains occupation-level data with `occ_code`, `title`, and `observed_exposure`.
- `task_penetration` contains task-level data with `task` and `penetration`.
- There is no occupation code or other common identifier in `task_penetration` that can be used to reliably connect individual tasks to occupations in `job_exposure`.

Therefore, the two datasets are kept as separate tables rather than forcing an unreliable merge. No merge-related row expansion, unmatched keys, or duplicate matches were introduced.

In [45]:
# Create cleaned copies so the original data stays unchanged
job_exposure_clean = job_exposure.copy()
task_penetration_clean = task_penetration.copy()

# Create a temporary standardized task key
task_key_clean = (
    task_penetration_clean["task"]
    .str.strip()
    .str.casefold()
)

# Remove repeated task records based on the standardized text
task_penetration_clean = (
    task_penetration_clean
    .loc[~task_key_clean.duplicated()]
    .reset_index(drop=True)
)

print("job_exposure:", len(job_exposure), "->", len(job_exposure_clean))
print("task_penetration:", len(task_penetration), "->", len(task_penetration_clean))

job_exposure: 756 -> 756
task_penetration: 17998 -> 17991


In [46]:
clean_task_key = (
    task_penetration_clean["task"]
    .str.strip()
    .str.casefold()
)

print("Rows removed:", len(task_penetration) - len(task_penetration_clean))
print("Repeated standardized tasks remaining:",
      clean_task_key.duplicated().sum())

print("Original penetration sum:",
      task_penetration["penetration"].sum())

print("Clean penetration sum:",
      task_penetration_clean["penetration"].sum())

Rows removed: 7
Repeated standardized tasks remaining: 0
Original penetration sum: 1201.2545999999998
Clean penetration sum: 1196.1704999999997


## 10. Reconcile Changes

**Status: Done**

I compared the original datasets with the cleaned versions to verify the effects of cleaning.

**Row counts:**
- `job_exposure`: 756 → 756 rows
- `task_penetration`: 17,998 → 17,991 rows

A total of 7 rows were removed from `task_penetration`. These rows were redundant copies of the same task identified during the duplicate and text-standardization checks. No rows were removed from `job_exposure`.

After cleaning:
- 0 repeated standardized task descriptions remain.
- The original `penetration` sum was approximately 1201.2546.
- The cleaned `penetration` sum is approximately 1196.1705.
- The difference is 5.0841, which equals 7 × 0.7263, matching the seven redundant records that were removed.

The change in the total is therefore fully explained by the duplicate removal, and no unexplained row or value changes were introduced.

## 11. Validate the Cleaned Data

**Status: Done**

I reran the key data-quality checks on the cleaned datasets.

**`job_exposure_clean`:**
- 756 rows
- 0 duplicate rows
- 0 repeated `occ_code` values
- 0 missing values
- 0 invalid `occ_code` formats
- 0 `observed_exposure` values outside the valid 0–1 range

**`task_penetration_clean`:**
- 17,991 rows
- 0 duplicate rows
- 0 repeated task descriptions after standardizing whitespace and capitalization
- 0 missing values
- 0 blank task descriptions
- 0 `penetration` values outside the valid 0–1 range

The cleaned datasets satisfy the cleaning rules established earlier in the notebook. No unresolved data-quality issues were identified in these validation checks.

In [47]:
# Validate cleaned job_exposure
print("job_exposure_clean")
print("Rows:", len(job_exposure_clean))
print("Duplicate rows:", job_exposure_clean.duplicated().sum())
print("Repeated occ_code:", job_exposure_clean["occ_code"].duplicated().sum())
print("Missing values:", job_exposure_clean.isna().sum().sum())
print("Invalid occ_code formats:",
      (~job_exposure_clean["occ_code"].str.match(r"^\d{2}-\d{4}$")).sum())
print("Exposure values outside 0-1:",
      ((job_exposure_clean["observed_exposure"] < 0) |
       (job_exposure_clean["observed_exposure"] > 1)).sum())


# Validate cleaned task_penetration
clean_task_key = (
    task_penetration_clean["task"]
    .str.strip()
    .str.casefold()
)

print("\ntask_penetration_clean")
print("Rows:", len(task_penetration_clean))
print("Duplicate rows:", task_penetration_clean.duplicated().sum())
print("Repeated standardized tasks:", clean_task_key.duplicated().sum())
print("Missing values:", task_penetration_clean.isna().sum().sum())
print("Blank tasks:",
      task_penetration_clean["task"].str.strip().eq("").sum())
print("Penetration values outside 0-1:",
      ((task_penetration_clean["penetration"] < 0) |
       (task_penetration_clean["penetration"] > 1)).sum())

job_exposure_clean
Rows: 756
Duplicate rows: 0
Repeated occ_code: 0
Missing values: 0
Invalid occ_code formats: 0
Exposure values outside 0-1: 0

task_penetration_clean
Rows: 17991
Duplicate rows: 0
Repeated standardized tasks: 0
Missing values: 0
Blank tasks: 0
Penetration values outside 0-1: 0


## 12. Save Clean Outputs

**Status: Done**

The cleaned datasets are saved as new CSV files so that the original source files remain unchanged.

**Cleaned outputs:**
- `data/job_exposure_clean.csv`
- `data/task_penetration_clean.csv`

The original CSV files are preserved separately and are not overwritten.

In [48]:
job_exposure_clean.to_csv(
    "data/job_exposure_clean.csv",
    index=False
)

task_penetration_clean.to_csv(
    "data/task_penetration_clean.csv",
    index=False
)

print("Cleaned files saved successfully.")

Cleaned files saved successfully.


## 13. Reproducibility Check

**Status: Done**

I restarted the notebook kernel and ran all cells from top to bottom.

The notebook successfully:
- Loaded the original source files.
- Reproduced the inspection and data-quality checks.
- Applied the documented cleaning rules.
- Recreated the cleaned datasets.
- Validated the final row counts and data-quality conditions.
- Saved the cleaned CSV files without overwriting the original source files.

The cleaning workflow is reproducible from the original data.

## 14. Document and Commit

**Status: Done**

The data-cleaning process and validation results are documented in this notebook.

**Data source:**  
Anthropic Economic Index, `labor_market_impacts`

**Cleaning summary:**
- The original source files were preserved and not overwritten.
- No missing values were found.
- No data type conversions were required.
- No invalid numeric values or occupation-code formats were found.
- Repeated task descriptions that differed only by capitalization and had identical penetration values were treated as the same task.
- 7 redundant task records were removed.
- No rows were removed from `job_exposure`.
- The cleaned datasets passed the final validation checks.

**Cleaned data locations:**
- `data/job_exposure_clean.csv`
- `data/task_penetration_clean.csv`

**Assumption:**  
Task descriptions that differ only in capitalization and have identical penetration values represent the same task.